# Inference Service Smoke Tests

Этот ноутбук проверяет, что FastAPI inference сервис работает и отвечает корректно.

**Проверки:**
- Health check на `/health`
- Успешное предсказание на `/predict`
- Validation error на некорректном входе
- Смена модели на `/change-model`

In [1]:
%load_ext autoreload
%autoreload 2

import os
import json
import requests

BASE_URL = os.getenv("INFERENCE_BASE_URL", "http://localhost:8000")
print(f"Using base URL: {BASE_URL}")

Using base URL: http://localhost:8000


## 1. Health Check

In [2]:
# Проверка работоспособности сервиса
r = requests.get(f"{BASE_URL}/health", timeout=10)
r.raise_for_status()
print(f"Status Code: {r.status_code}")
print(json.dumps(r.json(), indent=2))

Status Code: 200
{
  "status": "healthy",
  "model_loaded": true,
  "model_version": "v1"
}


## 2. Valid Prediction

In [3]:
# Тестовые данные для разных категорий
test_products = {
    "Electronics": "Sony wireless headphones with active noise cancellation and 30-hour battery life",
    "Books": "The Great Gatsby by F. Scott Fitzgerald, paperback edition, classic American literature",
    "Household": "Kitchen blender mixer 500W stainless steel with multiple speed settings",
    "Clothing & Accessories": "Men's cotton t-shirt casual wear blue color XL size, comfortable fit"
}

# Тестирование предсказаний
for category, description in test_products.items():
    payload = {"description": description}
    r = requests.post(f"{BASE_URL}/predict", json=payload, timeout=20)
    
    print(f"\n{'='*60}")
    print(f"Category: {category}")
    print(f"Description: {description[:50]}...")
    print(f"Status Code: {r.status_code}")
    
    if r.status_code == 200:
        result = r.json()
        print(f"Predicted: {result['category']} (confidence: {result['confidence']:.2%})")
        print(f"Model Version: {result['model_version']}")
    else:
        print(f"Error: {r.text}")


Category: Electronics
Description: Sony wireless headphones with active noise cancell...
Status Code: 200
Predicted: Electronics (confidence: 100.00%)
Model Version: v1

Category: Books
Description: The Great Gatsby by F. Scott Fitzgerald, paperback...
Status Code: 200
Predicted: Books (confidence: 100.00%)
Model Version: v1

Category: Household
Description: Kitchen blender mixer 500W stainless steel with mu...
Status Code: 200
Predicted: Household (confidence: 100.00%)
Model Version: v1

Category: Clothing & Accessories
Description: Men's cotton t-shirt casual wear blue color XL siz...
Status Code: 200
Predicted: Clothing & Accessories (confidence: 100.00%)
Model Version: v1


## 3. Invalid Payload - Validation Error

In [4]:
# Пустое описание (должно вызвать ошибку)
bad_payload = {"description": ""}

r = requests.post(f"{BASE_URL}/predict", json=bad_payload, timeout=20)
print(f"Empty description test:")
print(f"Status Code: {r.status_code}")
print(f"Response: {r.text}")

Empty description test:
Status Code: 422
Response: {"detail":[{"type":"string_too_short","loc":["body","description"],"msg":"String should have at least 1 character","input":"","ctx":{"min_length":1}}]}


In [5]:
# Отсутствие поля description (должно вызвать ошибку)
missing_field_payload = {"text": "Some text here"}

r = requests.post(f"{BASE_URL}/predict", json=missing_field_payload, timeout=20)
print(f"\nMissing field test:")
print(f"Status Code: {r.status_code}")
print(f"Response: {r.text[:200]}...")


Missing field test:
Status Code: 422
Response: {"detail":[{"type":"missing","loc":["body","description"],"msg":"Field required","input":{"text":"Some text here"}}]}...


## 4. List Available Models

In [6]:
# Получение списка доступных моделей
r = requests.get(f"{BASE_URL}/models", timeout=10)
print(f"Status Code: {r.status_code}")
print(f"Available Models: {json.dumps(r.json(), indent=2)}")

Status Code: 200
Available Models: [
  {
    "version": "v1",
    "is_current": true
  },
  {
    "version": "v2",
    "is_current": false
  }
]


## 5. Change Model (если есть несколько версий)

In [7]:
# Попытка сменить модель на v1
change_model_payload = {"version": "v1"}

r = requests.post(f"{BASE_URL}/change-model", json=change_model_payload, timeout=20)
print(f"Change model test:")
print(f"Status Code: {r.status_code}")

if r.status_code == 200:
    result = r.json()
    print(f"Success: {result['success']}")
    print(f"Message: {result['message']}")
    print(f"New Version: {result['model_version']}")
else:
    print(f"Response: {r.text}")

Change model test:
Status Code: 200
Success: True
Message: Модель версии v1 успешно загружена
New Version: v1


## 6. Invalid Model Version

In [8]:
# Попытка сменить на несуществующую версию
invalid_version_payload = {"version": "v99"}

r = requests.post(f"{BASE_URL}/change-model", json=invalid_version_payload, timeout=20)
print(f"Invalid version test:")
print(f"Status Code: {r.status_code}")
print(f"Response: {r.text}")

Invalid version test:
Status Code: 404
Response: {"detail":"Модель версии v99 не найдена. Доступные версии: ['v1', 'v2']"}


## 7. Root Endpoint

In [9]:
# Проверка корневого endpoint
r = requests.get(f"{BASE_URL}/", timeout=10)
print(f"Root endpoint:")
print(f"Status Code: {r.status_code}")
print(f"Response: {json.dumps(r.json(), indent=2)}")

Root endpoint:
Status Code: 200
Response: {
  "service": "E-commerce Classification API",
  "version": "1.0.0",
  "status": "running"
}


## 📊 Summary

| Test | Status |
|------|--------|
| Health Check | ✅ |
| Valid Predictions | ✅ |
| Validation Errors | ✅ |
| List Models | ✅ |
| Change Model | ✅ |
| Invalid Version | ✅ |
| Root Endpoint | ✅ |

**Все тесты пройдены!** Сервис работает корректно.